# **Kaggle API Setup and Dataset Download**

In [ ]:
print("Please upload your kaggle.json file!")
from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("\nDownloading the Brain MRI dataset (Small Version)...")
!kaggle datasets download -d navoneel/brain-mri-images-for-brain-tumor-detection

print("Unzipping dataset...")
!unzip -q brain-mri-images-for-brain-tumor-detection.zip

# Create Training, Validation, and Test Datasets

In [ ]:
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 32

train_dataset = tf.keras.utils.image_dataset_from_directory(
    'brain_tumor_dataset',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    'brain_tumor_dataset',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_batches = tf.data.experimental.cardinality(validation_dataset)
test_dataset = validation_dataset.take(1)
validation_dataset = validation_dataset.skip(1)

class_names = train_dataset.class_names
print("\nFound the following classes:", class_names)

# Build CNN Model with Data Augmentation

In [ ]:
from tensorflow.keras import layers, models

data_augmentation = models.Sequential(
  [
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
  ]
)

model = models.Sequential([
  layers.Rescaling(1./255, input_shape=(IMG_SIZE, IMG_SIZE, 3)),
  data_augmentation,
  layers.Conv2D(16, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(32, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Flatten(),
  layers.Dense(128, activation='relu'),
  layers.Dropout(0.5),
  layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

# Train, Visualize, and Evaluate Model

In [ ]:
epochs = 15
history = model.fit(
  train_dataset,
  validation_data=validation_dataset,
  epochs=epochs
)

import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(epochs)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

print("\nEvaluating model on the test set...")
loss, accuracy = model.evaluate(test_dataset)
print(f"\n Test Accuracy: {accuracy*100:.2f}%")

# Train the Model

In [ ]:
print("\nStarting model training... ")
epochs = 10
history = model.fit(
  train_dataset,
  validation_data=validation_dataset,
  epochs=epochs
)
print("\nModel training complete!")

# Visualize Predictions

In [ ]:
print("Evaluating model on the test set...")
loss, accuracy = model.evaluate(test_dataset)
print(f"\nTest Accuracy: {accuracy*100:.2f}%")
print("-" * 30)

print("The model will predict if an MRI has a tumor ('yes') or not ('no').\n")

plt.figure(figsize=(12, 12))

for images, labels in test_dataset.take(1):
  predictions = model.predict(images)

  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))

    predicted_class_index = 1 if predictions[i] > 0.5 else 0
    predicted_class_name = class_names[predicted_class_index]
    actual_class_name = class_names[int(labels[i])]

    title_color = 'green' if predicted_class_name == actual_class_name else 'red'

    plt.title(f"Actual: {actual_class_name}\nPredicted: {predicted_class_name}", color=title_color)
    plt.axis("off")

plt.tight_layout()
plt.show()